# Checking what a field actually says before I search it

*Part 1 of 3. Each part stands on its own, but together they follow one running example: building a cohort of female subjects with any diagnosis touching the circulatory system, searched across every source CDA covers.*

I study vascular involvement in cancer. For this cohort, I want female subjects with any diagnosis connected to the circulatory system -- searched across all six sources CDA covers, not one at a time. Before I write that filter, I want to check something: do these sources actually record `sex` the same way? If GDC uses `F` and another source spells out `Female`, a filter checking only one exact string could quietly miss real matches.

First, what's actually available to search on:

In [ ]:
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]
# Force light mode regardless of system/browser dark-mode preference --
# recent DataTables versions auto-detect prefers-color-scheme, which otherwise
# silently flips tables to a dark theme on dark-mode systems.
opt.css = """
.itables, .dataTable { color-scheme: light; }
"""

In [ ]:
from cdapython import columns
columns()

`sex` is there. Now let's check what values it actually takes -- one source at a time, since
`column_values()` only looks at a single source per call:

In [ ]:
from cdapython import column_values

gdc_values = column_values('sex', data_source='GDC')
gdc_values

In [ ]:
pdc_values = column_values('sex', data_source='PDC')
pdc_values

Comparing these two side by side tells me exactly what strings are in play in each source, before I commit to a filter. If I'd guessed instead of checking, I might have written a filter that quietly excluded a source that phrased things differently.

One thing worth knowing about `column_values()`: some columns -- IDs, especially -- can have an enormous number of distinct values, making this kind of check expensive to run. CDA will warn rather than run one of these by default. If I'm sure I actually want it, I can force it through:

In [ ]:
column_values('subject_id', data_source='GDC', force=True)

Now I can build a real cross-source query. `get_subject_data()` -- unlike `column_values()` -- does accept a list of sources in one call:

In [ ]:
from cdapython import get_subject_data

female_subjects = get_subject_data(
    match_all=['sex = F*'],
    data_source=['GDC', 'PDC', 'IDC', 'GC', 'ICDC', 'CTDC']
)
female_subjects

A couple of things worth noticing in this filter string: no quotes around the value, and a wildcard (`F*`) rather than one exact spelling, precisely because I just confirmed sources don't necessarily agree on how to write it out. The wildcard, not any special cross-source setting, is what's doing the work of catching both spellings here.

This is my starting cohort. In [Part 2](#), I narrow it down to circulatory-system diagnoses -- and it turns out that narrowing step has its own version of this same problem, solved a different way.

A matching `get_file_data()` function takes the same arguments, if file-level results are what I need instead of subject-level.